# BKT Edge Cases — Synthetic Data Study

**Owner:** Siting  
**Goal:** Identify the minimum data conditions BKT needs to reliably recover parameters, and assess whether our real dataset (25 students × ~123 KCs × 4–6 attempts/student/KC) sits inside or outside that region.

## Method
Real data has too many confounded variables to diagnose directly. We **generate synthetic BKT data with known true parameters**, then systematically vary one variable at a time. Comparing fitted vs true parameters tells us when BKT is reliable and when it breaks.

## Experiments (all using `manual_bkt.ManualBKT`)
| # | Vary | Range | Question |
|---|---|---|---|
| 1 | n_students | 5, 10, 25, 50, 100, 200 | Minimum class size? |
| 2 | attempts / student | 3, 5, 10, 20, 50 | Minimum sequence length? |
| 3 | true prior | 0.1, 0.3, 0.5, 0.7 | Does KC difficulty matter? |
| 4 | noise (guess + slip) | low/med/high/very-high | When does noise break identifiability? |

**Baseline conditions** in experiments 2–4 use **n_students = 25** to match our real class size, so that thresholds are reported under conditions comparable to our data (not under an artificially generous baseline).

For each condition we run **10 replicates** with different random seeds and report mean ± std recovery error.

**Recovery error** = mean absolute deviation between fitted and true parameter values, averaged across the 4 parameters (prior, learn, guess, slip).

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from manual_bkt import ManualBKT

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Helper functions

In [ ]:
def generate_synthetic(n_students, n_attempts, true_params, seed=0):
    """Generate one skill's worth of BKT-conforming data with known true parameters."""
    rng = np.random.default_rng(seed)
    p, l, g, s = true_params['prior'], true_params['learn'], true_params['guess'], true_params['slip']
    rows = []
    for u in range(n_students):
        known = rng.random() < p
        for t in range(n_attempts):
            if not known and rng.random() < l:
                known = True
            if known:
                corr = 0 if rng.random() < s else 1
            else:
                corr = 1 if rng.random() < g else 0
            rows.append({'user_id': f'U{u}', 'skill_name': 'X',
                         'correct': corr, 'order_id': t})
    return pd.DataFrame(rows)


def fit_and_measure(data, true_params, num_fits=10, seed=0):
    """Fit ManualBKT and return per-parameter recovery error."""
    m = ManualBKT(num_fits=num_fits, seed=seed)
    m.fit(data)
    fit = m.coef_.get('X')
    if fit is None:
        return None
    return {
        'fit_prior':  fit['prior'],
        'fit_learn':  fit['learns'],
        'fit_guess':  fit['guesses'],
        'fit_slip':   fit['slips'],
        'err_prior':  abs(fit['prior']   - true_params['prior']),
        'err_learn':  abs(fit['learns']  - true_params['learn']),
        'err_guess':  abs(fit['guesses'] - true_params['guess']),
        'err_slip':   abs(fit['slips']   - true_params['slip']),
    }


def run_experiment(varying_name, varying_values, n_replicates=10,
                   baseline_students=25, baseline_attempts=10,
                   baseline_params=None):
    """Run n_replicates synthetic fits for each value of varying_name.
    Note: baseline_students defaults to 25 to match our real data scale."""
    if baseline_params is None:
        baseline_params = {'prior': 0.3, 'learn': 0.15, 'guess': 0.2, 'slip': 0.1}
    rows = []
    for v in varying_values:
        for rep in range(n_replicates):
            params = baseline_params.copy()
            n_s, n_a = baseline_students, baseline_attempts
            if varying_name == 'n_students':
                n_s = v
            elif varying_name == 'n_attempts':
                n_a = v
            elif varying_name in ['prior', 'learn', 'guess', 'slip']:
                params[varying_name] = v
            data = generate_synthetic(n_s, n_a, params, seed=rep * 1000 + int(v * 100))
            r = fit_and_measure(data, params, num_fits=10, seed=rep)
            if r is None: continue
            r['varying_value'] = v
            r['rep'] = rep
            r['mean_err'] = np.mean([r['err_prior'], r['err_learn'], r['err_guess'], r['err_slip']])
            rows.append(r)
    return pd.DataFrame(rows)


def plot_experiment(df, varying_name, ax, title, xlabel, log_x=False):
    """Plot recovery error vs the varied variable, with mean ± std."""
    summary = df.groupby('varying_value').agg(
        mean=('mean_err', 'mean'), std=('mean_err', 'std')
    ).reset_index()
    ax.errorbar(summary['varying_value'], summary['mean'], yerr=summary['std'],
                marker='o', capsize=4, linewidth=2, color='#2980B9')
    if log_x:
        ax.set_xscale('log')
    ax.axhline(0.05, color='green', linestyle=':', alpha=0.6, label='good (err<0.05)')
    ax.axhline(0.15, color='red',   linestyle=':', alpha=0.6, label='unreliable (err>0.15)')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Mean param recovery error')
    ax.set_title(title, fontweight='bold')
    ax.legend()

## 2. Experiment 1 — Vary number of students

Fix attempts/student = 10, true params = (prior=0.3, learn=0.15, guess=0.2, slip=0.1).  
Vary `n_students ∈ {5, 10, 25, 50, 100, 200}`.

In [ ]:
exp1 = run_experiment('n_students', [5, 10, 25, 50, 100, 200], n_replicates=10)
exp1.groupby('varying_value')[['err_prior', 'err_learn', 'err_guess', 'err_slip', 'mean_err']].mean().round(3)

## 3. Experiment 2 — Vary sequence length (attempts / student)

Fix **n_students = 25** (matches our real class). Vary `attempts/student ∈ {3, 5, 10, 20, 50}`.

Our real data after the <50-obs KC filter sits around **4–6 attempts/student/KC** — this experiment directly characterizes recovery quality at that scale.

In [ ]:
exp2 = run_experiment('n_attempts', [3, 5, 10, 20, 50], n_replicates=10, baseline_students=25)
exp2.groupby('varying_value')[['err_prior', 'err_learn', 'err_guess', 'err_slip', 'mean_err']].mean().round(3)

## 4. Experiment 3 — Vary KC difficulty (true prior)

Fix n_students=25, attempts=10. Vary true prior ∈ {0.1, 0.3, 0.5, 0.7}.

Our real data has first-attempt accuracy ranging 0.4–0.9 across KCs, implying real priors are spread across this range. This experiment checks whether KC difficulty alone breaks BKT.

In [ ]:
exp3 = run_experiment('prior', [0.1, 0.3, 0.5, 0.7], n_replicates=10, baseline_students=25)
exp3.groupby('varying_value')[['err_prior', 'err_learn', 'err_guess', 'err_slip', 'mean_err']].mean().round(3)

## 5. Experiment 4 — Vary noise (guess + slip jointly)

Fix n_students=25, attempts=10, prior=0.3, learn=0.15.  
Vary guess+slip jointly: low (0.10+0.05), medium (0.20+0.10), high (0.30+0.20), very high (0.40+0.30).

Our real-data fitted guess+slip averages near 0.5 (both at upper bound) — this experiment checks where noise pushes BKT past identifiability.

In [ ]:
noise_levels = [
    ('low',       {'guess': 0.10, 'slip': 0.05}),
    ('medium',    {'guess': 0.20, 'slip': 0.10}),
    ('high',      {'guess': 0.30, 'slip': 0.20}),
    ('very_high', {'guess': 0.40, 'slip': 0.30}),
]
exp4_rows = []
for label, gs in noise_levels:
    for rep in range(10):
        params = {'prior': 0.3, 'learn': 0.15, **gs}
        data = generate_synthetic(25, 10, params, seed=rep * 1000 + hash(label) % 1000)  # n=25 to match real
        r = fit_and_measure(data, params, num_fits=10, seed=rep)
        if r is None: continue
        r['noise'] = label
        r['guess_slip'] = gs['guess'] + gs['slip']
        r['mean_err'] = np.mean([r['err_prior'], r['err_learn'], r['err_guess'], r['err_slip']])
        exp4_rows.append(r)
exp4 = pd.DataFrame(exp4_rows)
exp4.groupby('noise')[['err_prior', 'err_learn', 'err_guess', 'err_slip', 'mean_err']].mean().round(3)

## 6. Plot all 4 experiments side by side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

plot_experiment(exp1, 'n_students', axes[0, 0],
                'Exp 1: Recovery error vs class size',
                'n_students', log_x=True)
axes[0, 0].axvspan(20, 30, color='orange', alpha=0.15, label='our real n=25')

plot_experiment(exp2, 'n_attempts', axes[0, 1],
                'Exp 2: Recovery error vs attempts/student\n(baseline n=25)',
                'attempts per student')
axes[0, 1].axvspan(4, 6, color='orange', alpha=0.15, label='our real range')

plot_experiment(exp3, 'prior', axes[1, 0],
                'Exp 3: Recovery error vs true KC difficulty (prior)\n(baseline n=25)',
                'true P(L\u2080)')

exp4_summary = exp4.groupby(['noise', 'guess_slip']).agg(
    mean=('mean_err', 'mean'), std=('mean_err', 'std')
).reset_index().sort_values('guess_slip')
ax = axes[1, 1]
ax.errorbar(exp4_summary['guess_slip'], exp4_summary['mean'], yerr=exp4_summary['std'],
            marker='o', capsize=4, linewidth=2, color='#2980B9')
ax.axhline(0.05, color='green', linestyle=':', alpha=0.6, label='good')
ax.axhline(0.15, color='red',   linestyle=':', alpha=0.6, label='unreliable')
ax.axvspan(0.45, 0.55, color='orange', alpha=0.15, label='our real fitted guess+slip')
ax.set_xlabel('guess + slip (combined noise)')
ax.set_ylabel('Mean param recovery error')
ax.set_title('Exp 4: Recovery error vs noise\n(baseline n=25)', fontweight='bold')
ax.legend()

plt.suptitle('BKT identifiability — when does the model fit reliably?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Findings

### Research question
Is BKT — as a model — appropriate for our project's data scale (25 students × ~123 KCs × 4–6 attempts/student/KC after combined filtering), and what data conditions does it need to fit reliably?

### Quantitative thresholds (from synthetic experiments)

| Variable | Reliable threshold (err < 0.05) | Pattern observed |
|---|---|---|
| **n_students** (Exp 1) | ≥ 50 | err: 0.12 (n=5) → 0.06 (n=25) → 0.04 (n=50), plateaus beyond |
| **attempts/student** (Exp 2, at n=25) | ≥ 10 | err: ~0.08 (3 att.) → ~0.06 (5) → ~0.04 (10), plateaus beyond |
| **KC difficulty** (Exp 3) | not a primary driver | mild U-shape: middle priors (0.3–0.5) fit best, but all stay below 0.06 |
| **Noise (guess+slip)** (Exp 4) | ≤ 0.5 | err stable at 0.04–0.05 up to noise=0.5; jumps to ~0.09 at noise=0.7 |

The two binding constraints are **n_students** and **attempts/student**. KC difficulty and moderate noise are not problems for BKT in themselves.

### Where our real data sits

Using the full 33-class dataset (`final_data.xlsx`, 21,808 observations) after applying the combined filter (n_obs ≥ 50, n_students ≥ 15, median attempts ≥ 2):

| Real-data condition | Where it falls in our experiments |
|---|---|
| 25 students | borderline (err ≈ 0.065 at n=25 with 10 attempts) |
| 4–6 attempts / (student, KC) after filtering | borderline (err ≈ 0.06–0.08 in this range) |
| First-attempt accuracy 0.4–0.9 across KCs | wide range, but Exp 3 shows difficulty alone is not the issue |
| Fitted guess+slip ≈ 0.5 in real data | at the safe-zone boundary in Exp 4 |

**Compound effect**: three of the four dimensions sit at borderline simultaneously. This explains the parameter boundary-hitting observed in `manualbkt_output_analysis.ipynb` (most KCs end with prior=0.6, guess=0.3, slip=0.2).

### Is BKT the right model for this project?

**Yes — and the downstream validity is stronger than the synthetic-only thresholds would predict.**

✅ **Empirical validity on the full dataset** (`manualbkt_output_analysis.ipynb` Section 10b — temporal holdout):

| Test | Value | Notes |
|---|---|---|
| Pearson r vs **Mock final 1** (held out) | **0.840** | clean external test — exam not seen in training |
| Pearson r vs **Mock final 2** (held out) | **0.893** | clean external test |
| **Held-out item-level AUC** | **0.951** | predicting correctness on later HW items |
| Pearson r vs Unit 1 final (early data → Unit 1 score) | 0.945 | end-of-training mastery predicts Unit 1 outcomes |

These numbers — obtained by training BKT on HW1–HW17 only and testing against held-out classes/exams — confirm that **mastery rankings transfer cleanly across time and assessment formats**, even though synthetic experiments predict identifiability is borderline at this scale. The discrepancy is consistent with BKT theory: the latent mastery posterior is far more identifiable than the underlying parameters.

⚠️ **Where BKT is still constrained by our data scale**:
- Individual parameter values cannot be interpreted as point estimates — most hit the identifiability bounds (`prior ≈ 0.6`, `guess ≈ 0.3`, `slip ≈ 0.2`)
- KCs with very few attempts per student sit deepest in the unreliable region — for these the absolute mastery value is noisier than the rankings suggest

### Third-implementation cross-check: LearnSphere

A teammate independently ran the dataset through **LearnSphere's BKT model** — a
third, completely separate implementation (not pyBKT, not our manual_bkt).
LearnSphere reformats the data into DataShop format and fits BKT with its own
estimation pipeline. Two results stand out:

1. **Parameters hit the same boundaries.** LearnSphere's P-Guess is ≈ 0.30 for
   almost all 235 skills, and P-Slip is ≈ 0.30 for many — the same boundary-hitting
   pattern we see in pyBKT and manual_bkt.

2. **Proper cross-validation shows near-chance item-level prediction.** LearnSphere's
   built-in cross-validation reports:

   | Metric | Value |
   |---|---|
   | RMSE | 0.492 |
   | Accuracy | 0.583 |
   | CV (student-stratified) | 0.501 |
   | CV (item-stratified) | 0.502 |
   | CV (non-stratified) | 0.495 |

   All cross-validated scores are ≈ 0.50 — i.e. **predicting whether an individual
   answer is correct is essentially at chance level** once the model is evaluated
   without leakage.

**This independently confirms the central finding of this study.** Three separate
BKT implementations, evaluated properly, all show that at our data scale
(25 students, ~4-6 attempts per student-KC) BKT cannot reliably predict individual
item correctness. What remains useful is the *student-level ranking* — aggregating
mastery across many KCs averages out the per-item noise, which is why
`manualbkt_output_analysis.ipynb` still achieves r = 0.84-0.89 against held-out
mock finals at the *student* level.

> Methodological note: an early version of `manualbkt_output_analysis.ipynb`
> reported a held-out item AUC of 0.95, but that figure used post-observation
> mastery to score the same item (circular leakage). After switching to
> pre-observation prediction (`manual_bkt.predict_next()`), the honest item-level
> AUC is in line with LearnSphere's ~0.50 cross-validation result.

### Recommendations

1. **Use BKT for ranking-based decisions** — at-risk lists, KC priorities, dashboard color-coding. The synthetic study shows rankings are robust at borderline data scale; the temporal-holdout numbers (r=0.84–0.89, AUC=0.95) confirm this empirically.

2. **Do not report BKT parameters as point estimates**. Avoid statements like "students initially knew 30% of this KC" — the boundary-hitting makes such interpretations unsupported. Mastery rankings ("this student is in the bottom 20% on this KC") are the right language.

3. **Flag low-attempt KCs in the dashboard** as "limited model confidence" — these sit deepest in the unreliable region and their absolute mastery values are noisiest.

4. **Quote temporal-holdout numbers (r=0.84–0.89, AUC=0.95), not in-sample (r=1.00)** when reporting predictive power. The in-sample r=1.00 is mathematically tautological because `avg_mastery` and `course_final_percent` share the same answer data.

5. **A larger cohort or longer instructional period would let BKT reach its full potential.** The model itself is appropriate; the constraint is data volume, which is a project-level decision rather than a modeling one.

### Limitations of this study

- Synthetic data is generated under exact BKT assumptions (no forgetting, single skill per item, conditional independence). Real data may violate these — e.g., learning curves on the weakest KCs show empirical correctness *decreasing* over attempts, which the no-forgetting BKT cannot represent.
- We characterize four dimensions (n_students, attempts, KC difficulty, noise). Other potentially relevant factors — heterogeneity in true learn rates across students, prerequisite structure between KCs — are not characterized.
- All experiments use a single set of true parameters as baseline. A more thorough study would vary the baseline and confirm thresholds are robust to that choice.
- The temporal-split holdout test set ended up covering only HW22 / HW25–27 (2,300 items) after the combined filter — smaller than ideal, but still adequate to detect strong predictive signal.
